In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
#vars of interest
raw_vars <- read.table('ACE_autosomal_gnomAD_filtered_HQ_rare_CNVs_table_Children_included_in_analysis_v2024-04-29_sharing_2024-06-14.txt', sep = "\t")

# Make the first row as column names
colnames(raw_vars) <- raw_vars[1, ]

# Remove the first row
raw_vars <- raw_vars[-1, ]

#raw_vars$sampleID <- paste0(raw_vars$sampleID, "_", raw_vars$sampleID) 

In [3]:
colnames(raw_vars) <- c("chr", "start", "end", "svtype", "copy_number", "sampleID")

In [4]:
head(raw_vars)

,chr,start,end,svtype,copy_number,sampleID
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,chr6,167324974,167348151,DEL,1,AU3303303
3,chr1,11037596,11045765,DEL,1,AU4122301
4,chr1,1455363,1462806,DEL,1,AU3853302
5,chr6,49470062,49472956,DUP,3,AU3853302
6,chr7,5359301,5390726,DUP,3,AU3853302
7,chr7,5716619,6024899,DUP,3,AU3853302


In [4]:
raw_vars$change <- paste(raw_vars$chr, raw_vars$start, raw_vars$end, sep = ":")

In [5]:
raw_vars$chr <- as.numeric(gsub("chr", "", raw_vars$chr))

In [7]:
head(raw_vars)

,chr,start,end,svtype,copy_number,sampleID,change
,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,6,167324974,167348151,DEL,1,AU3303303,chr6:167324974:167348151
3,1,11037596,11045765,DEL,1,AU4122301,chr1:11037596:11045765
4,1,1455363,1462806,DEL,1,AU3853302,chr1:1455363:1462806
5,6,49470062,49472956,DUP,3,AU3853302,chr6:49470062:49472956
6,7,5359301,5390726,DUP,3,AU3853302,chr7:5359301:5390726
7,7,5716619,6024899,DUP,3,AU3853302,chr7:5716619:6024899


In [6]:
var_indv <- raw_vars %>%
  group_by(change) %>%
  summarize(sampleIDs = list(unique(sampleID)))

In [7]:
head(var_indv)

change,sampleIDs
<chr>,<list>
chr10:113583119:113731005,AU4935301
chr10:122488805:122566094,DG1015-302
chr10:122585174:122590792,AU1324303
chr10:122585174:122598110,DG1076-301
chr10:122597877:122600191,"AU4946302, AU4784303"
chr10:122832061:122850648,DG1069-301


In [8]:
dim(var_indv)

[1] 643   2

In [9]:
typeof(var_indv$sampleIDs)

[1] "list"

In [10]:
lengths_table <- sapply(var_indv$sampleIDs, length)

In [11]:
head(lengths_table)

[1] 1 1 1 1 2 1

In [10]:
table(lengths_table)

lengths_table
  1   2   3   4   5   6   7   9  11 
465 117  30  13  10   5   1   1   1 

In [12]:
var_info <- unique(raw_vars[,c("chr", "start", "end", "change")])

In [17]:
dim(var_info)

[1] 643   4

In [13]:
var_indv <- left_join(var_indv, var_info)

Joining with `by = join_by(change)`


In [19]:
dim(var_indv)

[1] 643   5

In [15]:
head(var_indv)

change,sampleIDs,chr,start,end
<chr>,<list>,<dbl>,<chr>,<chr>
chr10:113583119:113731005,AU4935301,10,113583119,113731005
chr10:122488805:122566094,DG1015-302,10,122488805,122566094
chr10:122585174:122590792,AU1324303,10,122585174,122590792
chr10:122585174:122598110,DG1076-301,10,122585174,122598110
chr10:122597877:122600191,"AU4946302, AU4784303",10,122597877,122600191
chr10:122832061:122850648,DG1069-301,10,122832061,122850648


In [14]:
var_indv$start <- as.numeric(var_indv$start)

var_indv$end <- as.numeric(var_indv$end)

In [17]:
c(outer(unlist(var_indv$sampleIDs[5]), c(".0", ".1"), paste0))

[1] "AU4946302.0" "AU4784303.0" "AU4946302.1" "AU4784303.1"

In [23]:
table(var_indv$chr)


 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 
73 49 34 28 27 22 32 27 29 29 35 31  7 12 31 35 25  9 61 18  6 23 

In [15]:
for(chr in 1:22){
    print(dim(var_indv[var_indv$chr == chr,]))
    }

[1] 73  5
[1] 49  5
[1] 34  5
[1] 28  5
[1] 27  5
[1] 22  5
[1] 32  5
[1] 27  5
[1] 29  5
[1] 29  5
[1] 35  5
[1] 31  5
[1] 7 5
[1] 12  5
[1] 31  5
[1] 35  5
[1] 25  5
[1] 9 5
[1] 61  5
[1] 18  5
[1] 6 5
[1] 23  5


In [59]:
msp_test_path <-  "/u/home/a/afcarrol/project-pasaniuc/Projects/20240423_ace_partial_hm3/rfmix_out/chr2.msp.tsv"
col_names <- readLines(msp_test_path, n = 2)[2]
col_names <- strsplit(col_names, "\t")[[1]]
col_names[1] <- substring(col_names[1], 2) 

In [60]:
test_msp <- read.table(msp_test_path, header = FALSE, sep = "\t", col.names = col_names)

In [61]:
dim(test_msp)

[1] 2001 5230

In [29]:
head(test_msp)

,chm,spos,epos,sgpos,egpos,n.snps,AU003305.0,AU003305.1,AU0226302.0,AU0226302.1,⋯,DG1580.201.0,DG1580.201.1,DG1580.202.0,DG1580.202.1,DG1580.301.0,DG1580.301.1,DG1228.202.0,DG1228.202.1,DG1228.302.0,DG1228.302.1
,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,2,15703,320574,0.00,0.24,132,0,0,0,0,⋯,0,1,0,0,1,0,1,1,1,1
2,2,320574,384236,0.24,0.43,20,0,0,0,0,⋯,0,1,0,0,1,0,1,1,1,1
3,2,384236,502023,0.43,0.70,40,0,0,0,0,⋯,0,1,0,0,1,0,1,1,1,1
4,2,502023,596532,0.70,0.75,40,0,0,0,0,⋯,0,1,0,0,1,0,1,1,1,1
5,2,596532,654472,0.75,0.86,50,0,0,0,0,⋯,0,1,0,0,1,0,1,1,1,1
6,2,654472,664719,0.86,0.93,10,0,0,0,0,⋯,0,1,0,0,1,0,1,1,1,1


In [36]:
# doesnt return all indv
find_regions <- function(chr_vars, msp){

# Initialize an empty dataframe to store the results
results_df <- data.frame(change = character(), zeros = integer(), 
                         ones = integer(), zero_ids = I(list()), one_ids = I(list()), 
                         stringsAsFactors = FALSE)
# Loop through each variant in chr_vars
for (i in seq_along(chr_vars$change)) {
  # Subset var_msp for the corresponding position
#i <- 6
  var_msp <- msp[msp$spos <= chr_vars$start[i] & msp$epos >= chr_vars$end[i], ]
    #print(var_msp[,1:4])
    #print(paste(chr_vars$start[i], chr_vars$end[i], sep = ": "))
  var_cols <- c(outer(unlist(chr_vars$sampleIDs[i]), c(".0", ".1"), paste0))
  #print(var_cols)
  var_msp <- var_msp[,colnames(var_msp) %in% var_cols]
  #print(colnames(var_msp))  
 if(nrow(var_msp)>1){
    print(paste("Overlapping Variant Window:", chr_vars$change[i]))}
     
 if(nrow(var_msp)==1){
    print(paste("Hit:", chr_vars$change[i]))
  
  # Extract columns ending with .0
 # id_columns <- grep(paste0(chr_vars$variant[i], "\\.0$"), names(var_msp), value = TRUE)
  id_columns <- grep("\\.0$", names(var_msp), value = TRUE)
  # Initialize counters for zeros and ones
  num_zeros <- 0
  num_ones <- 0
  zero_indv <- list()
  one_indv <- list()
      # Loop through each column and count zeros and ones
  for (col_name in id_columns) {
    # Get corresponding .1 column
    id_col_1 <- sub("\\.0$", ".1", col_name)
    
      print(var_msp)
    print(paste(col_name, var_msp[[col_name]], var_msp[[id_col_1]], sep = ": "))
  
    # Count zeros and ones
    if(var_msp[[col_name]] == 0 & var_msp[[id_col_1]] == 0){
        zero_indv <- c(zero_indv, list(sub("\\.0$", "", col_name)))
        num_zeros <- num_zeros + 1
    }
    if(var_msp[[col_name]] == 1 & var_msp[[id_col_1]] == 1){
        one_indv <- c(one_indv, list(sub("\\.0$", "", col_name)))
        num_ones <- num_ones + 1
    }
  }
  if(length(zero_indv)<1){
        zero_indv <- NA
  }
    if(length(one_indv)<1){
            one_indv <- NA
    }
 
  # Append results to the dataframe
  results_df <- rbind(results_df, data.frame(change = chr_vars$change[i], 
                                             zeros = num_zeros, ones = num_ones, 
                                             zero_ids = paste(unlist(zero_indv), sep=", "), 
                                             one_ids = paste(unlist(one_indv),sep=", ")))
 }
 
}
 return(results_df)
    }

In [63]:
find_regions <- function(chr_vars, msp) {

  # Initialize an empty dataframe to store the results
  results_df <- data.frame(change = character(), zeros = integer(), 
                           ones = integer(), zero_ids = I(list()), one_ids = I(list()), 
                           stringsAsFactors = FALSE)
  
  # Loop through each variant in chr_vars
  for (i in seq_along(chr_vars$change)) {
    # Subset var_msp for the corresponding position
    var_msp <- msp[msp$spos <= chr_vars$start[i] & msp$epos >= chr_vars$end[i], ]
    var_cols <- c(outer(unlist(chr_vars$sampleIDs[i]), c(".0", ".1"), paste0))
    var_msp <- var_msp[, colnames(var_msp) %in% var_cols]
    
    if (nrow(var_msp) > 1) {
      print(paste("Overlapping Variant Window:", chr_vars$change[i]))
    }
    
    if (nrow(var_msp) == 1) {
      print(paste("Hit:", chr_vars$change[i]))
      
      # Extract columns ending with .0
      id_columns <- grep("\\.0$", names(var_msp), value = TRUE)
      
      # Initialize counters for zeros and ones
      num_zeros <- 0
      num_ones <- 0
      zero_indv <- c()
      one_indv <- c()
      
      # Loop through each column and count zeros and ones
      for (col_name in id_columns) {
        # Get corresponding .1 column
        id_col_1 <- sub("\\.0$", ".1", col_name)
        
        # Count zeros and ones
        if (var_msp[[col_name]] == 0 & var_msp[[id_col_1]] == 0) {
          zero_indv <- c(zero_indv, sub("\\.0$", "", col_name))
          num_zeros <- num_zeros + 1
        }
        if (var_msp[[col_name]] == 1 & var_msp[[id_col_1]] == 1) {
          one_indv <- c(one_indv, sub("\\.0$", "", col_name))
          num_ones <- num_ones + 1
        }
      }
      
      if (length(zero_indv) < 1) {
        zero_indv <- NA
      }
      if (length(one_indv) < 1) {
        one_indv <- NA
      }
      
      # Append results to the dataframe
      results_df <- rbind(results_df, data.frame(
        change = chr_vars$change[i], 
        zeros = num_zeros, ones = num_ones, 
        zero_ids = I(list(zero_indv)), 
        one_ids = I(list(one_indv))
      ))
    }
  }
  return(results_df)
}


In [64]:
start.time <- Sys.time()
total_results <- data.frame(variant = character(), zeros = integer(), 
                         ones = integer(), zero_ids = I(list()), one_ids = I(list()), 
                         stringsAsFactors = FALSE)

msp_dir <- "/u/home/a/afcarrol/project-pasaniuc/Projects/20240423_ace_partial_hm3/rfmix_out/"
#chr <- 1
for(chr in 1:22){
    print(paste("Starting Chromosome:", chr))
    msp_path <- paste0(msp_dir, "chr", chr, ".msp.tsv")

    # Read the first line of the file to extract column names
    col_names <- readLines(msp_path, n = 2)[2]
    col_names <- strsplit(col_names, "\t")[[1]]
    col_names[1] <- substring(col_names[1], 2)  # Remove the leading #

    # Read the .tsv file into R, skipping the first line
    msp <- read.table(msp_path, header = FALSE, sep = "\t", col.names = col_names)
    
    chr_vars <- var_indv[var_indv$chr == chr,]
    
    chr_result <- find_regions(chr_vars, msp)
    
    total_results <- rbind(total_results, chr_result)
}
end.time <- Sys.time()
time.taken <- round(end.time - start.time,2)
time.taken

[1] "Starting Chromosome: 1"
[1] "Hit: chr1:103525603:103552876"
[1] "Hit: chr1:109235113:109238280"
[1] "Hit: chr1:109279459:109281327"
[1] "Hit: chr1:11029562:11045765"
[1] "Hit: chr1:11037596:11045765"
[1] "Hit: chr1:12755603:12776216"
[1] "Hit: chr1:12755603:12778004"
[1] "Hit: chr1:1455363:1462806"
[1] "Hit: chr1:1457032:1462806"
[1] "Hit: chr1:145730647:145789064"
[1] "Hit: chr1:153816176:153819833"
[1] "Hit: chr1:154947597:154948499"
[1] "Hit: chr1:155728945:155732299"
[1] "Hit: chr1:156339537:156347093"
[1] "Hit: chr1:160879214:160881619"
[1] "Hit: chr1:160948554:160952831"
[1] "Hit: chr1:168225841:168243309"
[1] "Hit: chr1:168231417:168236868"
[1] "Hit: chr1:169230617:169324598"
[1] "Hit: chr1:173533512:173858425"
[1] "Hit: chr1:186353591:186449581"
[1] "Hit: chr1:19201621:19260226"
[1] "Hit: chr1:196331059:196342347"
[1] "Hit: chr1:196993983:197004941"
[1] "Hit: chr1:197902533:197913278"
[1] "Hit: chr1:202434559:202440886"
[1] "Hit: chr1:206009167:206023993"
[1] "Hit: chr1:20

Time difference of 56.13 secs

In [65]:
head(total_results)


,change,zeros,ones,zero_ids,one_ids
,<chr>,<dbl>,<dbl>,<I<list>>,<I<list>>
1,chr1:103525603:103552876,0,0,NA,NA
2,chr1:109235113:109238280,0,0,NA,NA
3,chr1:109279459:109281327,0,1,NA,AU4991301
4,chr1:11029562:11045765,1,0,AU4963301,NA
5,chr1:11037596:11045765,0,1,NA,AU4891301
6,chr1:12755603:12776216,0,0,NA,NA


In [69]:
10926251>=109238280

[1] FALSE

In [66]:
dim(total_results)

[1] 412   5

In [39]:
table(total_results$ones)


  0   1   2   3   4   5   6 
262 119  52   6  20   5   6 

In [40]:
filtered <- total_results[total_results$zeros > 0 | total_results$ones > 0,]

In [41]:
dim(filtered)

[1] 236   5

In [43]:
total_results$EUR <- total_results$zero_ids

In [44]:
total_results$AFR <- total_results$one_ids

In [46]:
total_results$n_EUR <- total_results$zeros

In [47]:
total_results$n_AFR <- total_results$ones

In [48]:
head(var_indv)

change,sampleIDs,chr,start,end
<chr>,<list>,<dbl>,<dbl>,<dbl>
chr10:113583119:113731005,AU4935301,10,113583119,113731005
chr10:122488805:122566094,DG1015-302,10,122488805,122566094
chr10:122585174:122590792,AU1324303,10,122585174,122590792
chr10:122585174:122598110,DG1076-301,10,122585174,122598110
chr10:122597877:122600191,"AU4946302, AU4784303",10,122597877,122600191
chr10:122832061:122850648,DG1069-301,10,122832061,122850648


In [50]:
total_results <- left_join(total_results, var_indv[,c("change", "chr", "start", "end")])

Joining with `by = join_by(change)`


In [51]:
head(total_results)

,change,zeros,ones,zero_ids,one_ids,EUR,AFR,n_EUR,n_AFR,chr,start,end
,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:103525603:103552876,0,0,NA,NA,NA,NA,0,0,1,103525603,103552876
2,chr1:109235113:109238280,0,0,NA,NA,NA,NA,0,0,1,109235113,109238280
3,chr1:109279459:109281327,0,1,NA,AU4991301,NA,AU4991301,0,1,1,109279459,109281327
4,chr1:11029562:11045765,1,0,AU4963301,NA,AU4963301,NA,1,0,1,11029562,11045765
5,chr1:11037596:11045765,0,1,NA,AU4891301,NA,AU4891301,0,1,1,11037596,11045765
6,chr1:12755603:12776216,0,0,NA,NA,NA,NA,0,0,1,12755603,12776216


In [53]:
final_df <- total_results[,c("change", "chr", "start", "end", "n_EUR", "n_AFR", "EUR", "AFR")]

In [52]:
write_csv(total_results, "ad3_window.csv")

In [54]:
head(final_df)

,change,chr,start,end,n_EUR,n_AFR,EUR,AFR
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,chr1:103525603:103552876,1,103525603,103552876,0,0,NA,NA
2,chr1:109235113:109238280,1,109235113,109238280,0,0,NA,NA
3,chr1:109279459:109281327,1,109279459,109281327,0,1,NA,AU4991301
4,chr1:11029562:11045765,1,11029562,11045765,1,0,AU4963301,NA
5,chr1:11037596:11045765,1,11037596,11045765,0,1,NA,AU4891301
6,chr1:12755603:12776216,1,12755603,12776216,0,0,NA,NA


In [56]:
write_csv(final_df, "ad3_info.csv")

In [57]:
sum(final_df$n_EUR)

[1] 38

In [58]:
sum(final_df$n_AFR)

[1] 382

In [25]:
var_2 <- var_indv[var_indv$chr == 2,]

In [26]:
head(var_2)

change,sampleIDs,chr,start,end
<chr>,<list>,<dbl>,<chr>,<chr>
chr2:10052130:10211823,AU4591301,2,10052130,10211823
chr2:110115591:110212973,DG1246-301,2,110115591,110212973
chr2:111940205:112029659,AU4734301,2,111940205,112029659
chr2:120127591:120294808,DG1158-301,2,120127591,120294808
chr2:127279076:127324339,AU3128302,2,127279076,127324339
chr2:130313837:130375503,DG1040-302,2,130313837,130375503


In [30]:
find_regions(var_2, test_msp)

[1] "Overlapping variant window: chr2:110115591:110212973"
[1] "Overlapping variant window: chr2:111940205:112029659"
[1] "Overlapping variant window: chr2:120127591:120294808"
[1] "Overlapping variant window: chr2:127279076:127324339"
[1] "Overlapping variant window: chr2:130313837:130375503"
[1] "Overlapping variant window: chr2:130330045:130375503"
[1] "Overlapping variant window: chr2:130728973:131147440"
[1] "Overlapping variant window: chr2:130837358:131147440"
[1] "Overlapping variant window: chr2:147917186:147948322"
[1] "Overlapping variant window: chr2:147972642:147976073"
[1] "Overlapping variant window: chr2:189441349:189488090"
[1] "Overlapping variant window: chr2:189741245:189755658"
[1] "Overlapping variant window: chr2:199935834:199940231"
[1] "Overlapping variant window: chr2:206745525:206749281"
[1] "Overlapping variant window: chr2:208170848:208187159"
[1] "Overlapping variant window: chr2:217803770:217831645"
[1] "Overlapping variant window: chr2:227080346:22714768

variant,zeros,ones
<chr>,<dbl>,<dbl>
chr2:10052130:10211823,0,0
chr2:110115591:110212973,0,0
chr2:111940205:112029659,0,3
chr2:120127591:120294808,0,0
chr2:127279076:127324339,0,3
chr2:130313837:130375503,0,0
chr2:130330045:130375503,0,0
chr2:130728973:131147440,0,0
chr2:130837358:131147440,0,2
